# Test dataset 크기 Slicing

In [ ]:
pip install dipy

In [ ]:
# 모듈 

import nibabel as nib
import os 
import numpy as np

# 
from os.path import join as pjoin
import numpy as np
from dipy.viz import regtools
from dipy.data import fetch_stanford_hardi
from dipy.data.fetcher import fetch_syn_data
from dipy.io.image import load_nifti
from dipy.align.imaffine import (transform_centers_of_mass,
                                 AffineMap,
                                 MutualInformationMetric,
                                 AffineRegistration)
from dipy.align.transforms import (TranslationTransform3D,
                                   RigidTransform3D,
                                   AffineTransform3D)

In [ ]:
import nibabel as nib
from skimage.transform import resize
import itertools

In [3]:
data_dir = "./BTCV_Data/3affine"
data_list = sorted(os.listdir(data_dir))

In [41]:
# 2windowing 을 3 resize로 
a = 10

for i in range (0, 40):
    filename1 = "./RawData/Training/label/label00" + str(a) + ".nii.gz"
    filename2 = "./RawData/Training/img/img00" + str(a) + ".nii.gz"
    
    
    if os.path.isfile(filename1):
        label = nib.load(filename1).get_fdata()
        img = nib.load(filename2).get_fdata()

        h,w,d = label.shape
        
        label_start = 0
        label_end = 0
        label_buffer = 0
        cnt_class = 0
        for dep in range(40,d):
            for hei in range(0,h):
                for wid in range(0,w):
                    cnt_class += label[hei][wid][dep]
                    
                    if label_buffer == 0 and label[hei][wid][dep]!= 0:
                        label_start = dep
                        label_buffer = 1
                        print(str(a) + ":" + str(label_start))

            if label_buffer == 1 and cnt_class == 0:
                label_end = dep
                label_buffer = 2
                print(str(a) + ":" + str(label_end))

            elif dep == d-1:
                label_end = dep
                print(str(a) + ":" + str(label_end))
            
            if label_buffer ==2:
                break
                    
            cnt_class = 0
        
        
        
        label = label[:,:,label_start:label_end]
        label = resize_data(label)
        
        img = img[:,:,label_start:label_end]
        img = resize_data(img)
        
        savename1 = "./BTCV_Data/gt_resize/" + str(a) + ".nii.gz"
        savename2 = "./BTCV_Data/1resize/" + str(a) + ".nii.gz"
        
        
        x1 = nib.Nifti1Image(label, None) 
        nib.save(x1, savename1)
        x2 = nib.Nifti1Image(img, None) 
        nib.save(x2, savename2)
        print("clear")
        
        
#         print(a, ":", x.shape )
    
    a+=1
    
    
    

10:62
10:147
clear
21:75
21:142
clear
22:41
22:88
clear
23:45
23:95
clear
24:62
24:123
clear
25:40
25:84
clear
26:45
26:124
clear
27:40
27:87
clear
28:40
28:88
clear
29:40
29:99
clear
30:78
30:152
clear
31:40
31:92
clear
32:69
32:143
clear
33:44
33:103
clear
34:45
34:97
clear
35:46
35:93
clear
36:86
36:183
clear
37:52
37:98
clear
38:41
38:99
clear
39:46
39:89
clear
40:68
40:180
clear


In [4]:
st = []
a=1
for i in range (0, 40):
    filename = "./BTCV_Data/3affine/" + str(a) + ".nii.gz"
    if os.path.isfile(filename):
        x = nib.load(filename).get_fdata()
        x = x.swapaxes(0,2)
        st.append(x)
        
        
    a+=1  
st = np.array(st)
print(st.shape)

savename = "./BTCV_Data/affine_exis_stack.nii.gz"

x = nib.Nifti1Image(st, None) 
nib.save(x, savename)

(29, 64, 128, 128)


In [5]:
import os

In [8]:
# a = dcm.dcmread('CHAOS_Train_Sets/Train_Sets/CT/2/DICOM_anon/i0000,0000b.dcm')
# print(type(a.pixel_array))
data_dir = 'RawData/Training/img'

In [37]:
# Resize
def resize_data(data):
    initial_size_x = data.shape[0]
    initial_size_y = data.shape[1]
    initial_size_z = data.shape[2]

    new_size_x = 128
    new_size_y = 128
    new_size_z = 64

    delta_x = initial_size_x / new_size_x
    delta_y = initial_size_y / new_size_y
    delta_z = initial_size_z / new_size_z

    new_data = np.zeros((new_size_x, new_size_y, new_size_z))

    for x, y, z in itertools.product(range(new_size_x),
                                     range(new_size_y),
                                     range(new_size_z)):
        new_data[x][y][z] = data[int(x * delta_x)][int(y * delta_y)][int(z * delta_z)]

    return new_data